# Generación continua de lecturas de sensores — Elasticsearch directo (`sensores-servidores`)

Este notebook simula un flujo **near real-time**: cada 5 segundos genera una lectura por servidor y las indexa directamente en Elasticsearch con `bulk` (sin pasar por Logstash).

**Requisitos:**
```
pip install Faker elasticsearch
```

In [1]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import random
import time
from datetime import datetime, timezone
from faker import Faker

fake = Faker()

## Conexión a Elasticsearch

In [2]:
es = Elasticsearch("http://localhost:9200")

# Verificación rápida de conexión
print("Conectado:", es.ping())
print(es.info())

Conectado: True
{'name': 'debf2fb218ef', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'Alf65tuBQW2DDNzABjD2DQ', 'version': {'number': '7.17.10', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'fecd68e3150eda0c307ab9a9d7557f5d5fd71349', 'build_date': '2023-04-23T05:33:18.138275597Z', 'build_snapshot': False, 'lucene_version': '8.11.1', 'minimum_wire_compatibility_version': '6.8.0', 'minimum_index_compatibility_version': '6.0.0-beta1'}, 'tagline': 'You Know, for Search'}


## Servidores simulados

Cada "servidor" representa una instancia de un juego conocido del dataset de Steam, identificado por su `appid` real.

In [3]:
SERVIDORES = [
    {"appid": 730,     "nombre_juego": "Counter-Strike 2",       "servidor_id": "cs2-eu-01"},
    {"appid": 578080,  "nombre_juego": "PUBG: BATTLEGROUNDS",    "servidor_id": "pubg-na-01"},
    {"appid": 570,     "nombre_juego": "Dota 2",                 "servidor_id": "dota2-sa-01"},
    {"appid": 271590,  "nombre_juego": "Grand Theft Auto V",     "servidor_id": "gta5-eu-02"},
    {"appid": 1091500, "nombre_juego": "Cyberpunk 2077",         "servidor_id": "cp2077-na-01"},
]

ESTADOS_POSIBLES = ["online", "online", "online", "degradado", "mantenimiento"]

## Generador de lecturas de sensor

Misma idea que en los notebooks de CSV y MySQL: los campos de dominio controlado (`appid`, `estado_servidor`) se mantienen fijos/coherentes con el proyecto, mientras que los campos de variabilidad "de red" (`host`, `ip_origen`) se generan con Faker en cada lectura.

In [4]:
def generar_lectura(servidor):
    estado = random.choice(ESTADOS_POSIBLES)

    # Si el servidor está en mantenimiento, no tiene sentido tener jugadores activos ni CPU alta
    if estado == "mantenimiento":
        jugadores_activos = 0
        uso_cpu = round(random.uniform(1, 5), 2)
        latencia_ms = None
    else:
        jugadores_activos = random.randint(50, 5000)
        # Un estado 'degradado' implica más carga y más latencia
        uso_cpu = round(random.uniform(60, 95), 2) if estado == "degradado" else round(random.uniform(15, 70), 2)
        latencia_ms = random.randint(80, 400) if estado == "degradado" else random.randint(15, 90)

    return {
        "appid": servidor["appid"],
        "nombre_juego": servidor["nombre_juego"],
        "servidor_id": servidor["servidor_id"],
        "host": fake.hostname(),
        "ip_origen": fake.ipv4(),
        "estado_servidor": estado,
        "jugadores_activos": jugadores_activos,
        "uso_cpu_pct": uso_cpu,
        "latencia_ms": latencia_ms,
        "fuente": "python_sensores_directo",
        "@timestamp": datetime.now(timezone.utc).isoformat(),
    }

## Nombre del índice

Igual que en `mysql_logstash.conf`, usamos un índice diario para mantener consistencia entre fuentes.

In [5]:
def nombre_indice():
    fecha = datetime.now().strftime("%Y.%m.%d")
    return f"sensores-servidores-{fecha}"

## Bucle de simulación (near real-time)

Cada **5 segundos**, genera una lectura para **cada servidor** (5 documentos por ciclo) y los indexa en Elasticsearch usando `bulk` (más eficiente que indexar uno por uno). Detén la celda (interrumpir kernel) para parar la simulación.

In [6]:
INTERVALO_SEGUNDOS = 5

print("Enviando lecturas de sensores cada", INTERVALO_SEGUNDOS, "segundos. Interrumpe el kernel para detener.\n")

ciclo = 0
total_docs = 0
try:
    while True:
        ciclo += 1
        indice = nombre_indice()
        lecturas = [generar_lectura(s) for s in SERVIDORES]

        acciones = [
            {"_index": indice, "_source": lectura}
            for lectura in lecturas
        ]

        exitosos, errores = bulk(es, acciones)
        total_docs += exitosos

        print(f"[Ciclo {ciclo}] {exitosos} documentos indexados en '{indice}' (total acumulado: {total_docs})")
        for l in lecturas:
            print(f"   - {l['servidor_id']}: estado={l['estado_servidor']}, "
                  f"jugadores={l['jugadores_activos']}, cpu={l['uso_cpu_pct']}%, "
                  f"latencia={l['latencia_ms']}ms")

        time.sleep(INTERVALO_SEGUNDOS)
except KeyboardInterrupt:
    print("\nSimulación detenida por el usuario.")
    print(f"Total de documentos indexados en esta sesión: {total_docs}")

Enviando lecturas de sensores cada 5 segundos. Interrumpe el kernel para detener.

[Ciclo 1] 5 documentos indexados en 'sensores-servidores-2026.09.06' (total acumulado: 5)
   - cs2-eu-01: estado=online, jugadores=3404, cpu=69.08%, latencia=78ms
   - pubg-na-01: estado=mantenimiento, jugadores=0, cpu=1.08%, latencia=Nonems
   - dota2-sa-01: estado=online, jugadores=1806, cpu=65.27%, latencia=59ms
   - gta5-eu-02: estado=online, jugadores=1958, cpu=67.68%, latencia=23ms
   - cp2077-na-01: estado=degradado, jugadores=2223, cpu=80.11%, latencia=142ms
[Ciclo 2] 5 documentos indexados en 'sensores-servidores-2026.09.06' (total acumulado: 10)
   - cs2-eu-01: estado=online, jugadores=2968, cpu=43.05%, latencia=48ms
   - pubg-na-01: estado=online, jugadores=4573, cpu=21.34%, latencia=25ms
   - dota2-sa-01: estado=online, jugadores=1020, cpu=41.77%, latencia=71ms
   - gta5-eu-02: estado=online, jugadores=1525, cpu=19.64%, latencia=74ms
   - cp2077-na-01: estado=mantenimiento, jugadores=0, cpu=2

ConnectionError: Connection error caused by: ConnectionError(Connection error caused by: NewConnectionError(HTTPConnection(host='localhost', port=9200): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it))